# A · Paired test (same shirt)

- each test person wears their own shirt, so the real photo is the right answer
- produces: SSIM, LPIPS, seconds per image, worst 8 and best 8 grids
- your own Colab account · **Runtime → Change runtime type → T4 GPU**, then Run all
- output: `MyDrive/viton/results_A/`

## 1. Settings

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

ROLE = 'A'
GUIDANCE = 2.5
H, W = 512, 384
SEED = 42
DEV = 'cuda'
SAVE_TO_DRIVE = True  # keeps outputs if colab disconnects
N_PAIRED = 200
STEPS = 30

## 2. Drive

In [ ]:
import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = f'/content/drive/MyDrive/viton/results_{ROLE}'
else:
    OUT = f'/content/results_{ROLE}'
os.makedirs(OUT, exist_ok=True)
print('outputs ->', OUT)

## 3. Install and get CatVTON code

In [ ]:
!pip -q install diffusers accelerate transformers peft torchmetrics torch-fidelity lpips gdown gradio
!rm -rf /content/CatVTON
!GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 https://huggingface.co/spaces/zhengchong/CatVTON /content/CatVTON

import sys
sys.path.insert(0, '/content/CatVTON')
os.chdir('/content/CatVTON')
print(sorted(os.listdir('.')))

## Get VITON-HD

- official link from the VITON-HD repo (google drive)
- if gdown fails with a quota error: open the link in a browser, add it to your drive, put the zip at `MyDrive/viton/zalando-hd-resized.zip`, run this cell again

In [ ]:
import glob, shutil
ZIP = '/content/viton.zip'
DRIVE_ZIP = '/content/drive/MyDrive/viton/zalando-hd-resized.zip'

if not os.path.exists(ZIP):
    if os.path.exists(DRIVE_ZIP):
        shutil.copy(DRIVE_ZIP, ZIP)
    else:
        !gdown 1tLx8LRp-sxDp0EcYmYoV_vXdSc-jJ79w -O {ZIP}

size_gb = os.path.getsize(ZIP) / 1e9 if os.path.exists(ZIP) else 0
print(f'zip: {size_gb:.2f} GB')
assert size_gb > 1, 'download failed, see the note above this cell'

!unzip -q -o {ZIP} -x '*train/*' -d /content/vitonhd

TEST = os.path.dirname(glob.glob('/content/vitonhd/**/test/image', recursive=True)[0])
PAIRS = glob.glob('/content/vitonhd/**/test_pairs.txt', recursive=True)[0]
print('test folder:', TEST)
print('subfolders:', sorted(os.listdir(TEST)))
!head -3 {PAIRS}

## 5. Masks

In [ ]:
import numpy as np, cv2, random, time, json
from PIL import Image
import matplotlib.pyplot as plt

def mask_from_agnostic(name):
    img = np.asarray(Image.open(f'{TEST}/image/{name}').convert('RGB')).astype(int)
    agn = np.asarray(Image.open(f'{TEST}/agnostic-v3.2/{name}').convert('RGB')).astype(int)
    m = (np.abs(img - agn).sum(2) > 40).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    m = cv2.dilate(m, np.ones((9, 9), np.uint8))
    return Image.fromarray(m * 255)

def mask_from_parse(name):
    p = np.asarray(Image.open(f'{TEST}/image-parse-v3/{name.replace(".jpg", ".png")}'))
    m = np.isin(p, [5, 6, 7, 14, 15]).astype(np.uint8)  # upper clothes, dress, coat, arms
    m = cv2.dilate(m, np.ones((25, 25), np.uint8))
    return Image.fromarray(m * 255)

USE_AGNOSTIC = os.path.isdir(f'{TEST}/agnostic-v3.2')
get_mask = mask_from_agnostic if USE_AGNOSTIC else mask_from_parse
print('mask source:', 'agnostic-v3.2' if USE_AGNOSTIC else 'image-parse-v3')

names = sorted(os.listdir(f'{TEST}/image'))
with open(PAIRS) as f:
    pair_lines = [l.split() for l in f if l.strip()]
print(len(names), 'test people,', len(pair_lines), 'unpaired pairs')

def show_row(ax, imgs, titles):
    for a, im, t in zip(ax, imgs, titles):
        a.imshow(im, cmap='gray'); a.set_title(t, fontsize=9); a.axis('off')

fig, ax = plt.subplots(2, 4, figsize=(12, 8))
for i, n in enumerate(names[:4]):
    ax[0, i].imshow(Image.open(f'{TEST}/image/{n}')); ax[0, i].set_title(n); ax[0, i].axis('off')
    ax[1, i].imshow(get_mask(n), cmap='gray'); ax[1, i].axis('off')
plt.tight_layout(); plt.show()

## 6. Load CatVTON

In [ ]:
import torch, types, functools, tqdm as _tqdm
from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor
import model.pipeline as mp
from model.pipeline import CatVTONPipeline
from utils import resize_and_crop, resize_and_padding

mp.tqdm = types.SimpleNamespace(tqdm=functools.partial(_tqdm.tqdm, disable=True))  # quiet per-step bars

REPO = snapshot_download('zhengchong/CatVTON', allow_patterns=['vitonhd-16k-512/*', 'DensePose/*', 'SCHP/*'])
pipe = CatVTONPipeline(
    base_ckpt='booksforcharlie/stable-diffusion-inpainting',
    attn_ckpt=REPO,
    attn_ckpt_version='vitonhd',
    weight_dtype=torch.float16,
    device='cuda',
    skip_safety_check=True,
    use_tf32=False,
)
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)

def tryon(person, cloth, mask, steps=30, seed=SEED):
    person = resize_and_crop(person.convert('RGB'), (W, H))
    mask = resize_and_crop(mask.convert('L'), (W, H))
    cloth = resize_and_padding(cloth.convert('RGB'), (W, H))
    mask = mask_processor.blur(mask, blur_factor=9)
    g = torch.Generator(device='cuda').manual_seed(int(seed))
    out = pipe(image=person, condition_image=cloth, mask=mask, num_inference_steps=int(steps),
               guidance_scale=GUIDANCE, height=H, width=W, generator=g)[0]
    return out, person, cloth, mask

print('gpu memory used: %.1f GB' % (torch.cuda.memory_allocated() / 1e9))

## 7. Run try-on

- finished images are skipped, so after a disconnect just run again

In [ ]:
def out_path(kind, p, c):
    return f'{OUT}/{kind}/{p.replace(".jpg", "")}__{c.replace(".jpg", "")}.png'

def run_jobs(jobs, steps):
    todo = [j for j in jobs if not os.path.exists(out_path(*j))]
    print(f'{len(jobs)} jobs, {len(todo)} left')
    t0 = time.time()
    for i, (kind, p, c) in enumerate(todo):
        os.makedirs(f'{OUT}/{kind}', exist_ok=True)
        out, *_ = tryon(Image.open(f'{TEST}/image/{p}'), Image.open(f'{TEST}/cloth/{c}'), get_mask(p), steps=steps)
        out.save(out_path(kind, p, c))
        if i == 2 or (i + 1) % 25 == 0:
            per = (time.time() - t0) / (i + 1)
            print(f'{i + 1}/{len(todo)}  {per:.1f} s/img  ~{per * (len(todo) - i - 1) / 60:.0f} min left')
    per = (time.time() - t0) / max(len(todo), 1)
    print('done in %.1f min' % ((time.time() - t0) / 60))
    return per if todo else None

In [ ]:
paired = random.Random(0).sample(names, N_PAIRED)
sec_per_img = run_jobs([('paired', n, n) for n in paired], STEPS)

## 8. SSIM and LPIPS

- SSIM: higher is better
- LPIPS: lower is better

In [ ]:
import pandas as pd, torch
import torchvision.transforms.functional as TF
from torchmetrics.functional.image import structural_similarity_index_measure as ssim_fn
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

def to_t(im):
    return TF.to_tensor(im.convert('RGB')).unsqueeze(0).to(DEV)

def real_photo(p):
    return resize_and_crop(Image.open(f'{TEST}/image/{p}'), (W, H))

lpips = LearnedPerceptualImagePatchSimilarity(net_type='alex', normalize=True).to(DEV)
rows = []
for f in sorted(os.listdir(f'{OUT}/paired')):
    p = f.split('__')[0]
    fake, real = to_t(Image.open(f'{OUT}/paired/{f}')), to_t(real_photo(f'{p}.jpg'))
    rows.append({'person': p, 'ssim': ssim_fn(fake, real, data_range=1.0).item(), 'lpips': lpips(fake, real).item()})
per_image = pd.DataFrame(rows)
per_image.to_csv(f'{OUT}/paired_per_image.csv', index=False)

metrics = {'role': 'A paired', 'model': 'CatVTON vitonhd-16k-512', 'size': f'{W}x{H}', 'steps': STEPS,
           'n_paired': len(per_image), 'ssim': round(per_image.ssim.mean(), 4), 'lpips': round(per_image.lpips.mean(), 4),
           'sec_per_img_t4': round(sec_per_img, 2) if sec_per_img else None}
json.dump(metrics, open(f'{OUT}/metrics_A.json', 'w'), indent=2)
pd.Series(metrics)

## 9. Worst 8 and best 8

- columns: person, shirt, mask, try-on, real photo

In [ ]:
def grid(df, title, path):
    fig, ax = plt.subplots(len(df), 5, figsize=(13, 3.6 * len(df)))
    for r, (_, row) in enumerate(df.iterrows()):
        p = row.person
        show_row(ax[r], [real_photo(f'{p}.jpg'),
                         resize_and_padding(Image.open(f'{TEST}/cloth/{p}.jpg').convert('RGB'), (W, H)),
                         resize_and_crop(get_mask(f'{p}.jpg'), (W, H)),
                         Image.open(f'{OUT}/paired/{p}__{p}.png'),
                         real_photo(f'{p}.jpg')],
                 ['person', 'shirt', 'mask', f'try-on  lpips {row.lpips:.3f}', 'real'])
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path, dpi=100); plt.show()

grid(per_image.nlargest(8, 'lpips'), 'worst 8 by LPIPS', f'{OUT}/worst8.png')
grid(per_image.nsmallest(8, 'lpips'), 'best 8 by LPIPS', f'{OUT}/best8.png')

## 10. Summary

In [ ]:
print(json.dumps(metrics, indent=2))
print('saved in', OUT)
!ls {OUT}